In [ ]:
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt
import torch
import albumentations as A
import numpy as np
import shutil

# TODO: Update this path to your dataset location
data_path = "./data/cleaned_data"

In [ ]:
# TODO: Update this path to your trained model location
model = YOLO('../runs/detect/mahjong-yolom_94epochs/weights/best.pt') 

metrics = model.val()
 
metrics 

In [ ]:

# 定义保存预测结果的目录
output_dir = data_path + '/predicted'
os.makedirs(output_dir, exist_ok=True)

# 遍历根目录下的所有子文件夹
for subdir, _, files in os.walk(data_path):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(subdir, file)
            # 对图片进行预测
            results = model.predict(source=img_path, save_txt=True, save_conf=True, project=output_dir,     
                                    conf=0.5,  # 设置置信度阈值 (0.5 是示例，可根据需求调整)
                                    iou=0.45)
            # 移动保存的预测结果到指定目录
            label_dir = os.path.join(output_dir, 'labels')
            if os.path.exists(label_dir):
                for label_file in os.listdir(label_dir):
                    shutil.move(os.path.join(label_dir, label_file), output_dir)
                os.rmdir(label_dir)

# 使用验证集生成 COCO 格式的预测结果
coco_metrics = model.val(save_json=True)